# Home task 

1) Replicate Simple recommender implementation 
1) (optional) Replicate the the content based  recommender implementation 

[Beginner Tutorial: Recommender Systems in Python]
(https://www.datacamp.com/community/tutorials/recommender-systems-python?utm_source=adwords_ppc&utm_campaignid=1455363063&utm_adgroupid=65083631748&utm_device=c&utm_keyword=&utm_matchtype=b&utm_network=g&utm_adpostion=&utm_creative=332602034358&utm_targetid=aud-748597547652:dsa-473406569915&utm_loc_interest_ms=&utm_loc_physical_ms=1012865&gclid=Cj0KCQjwsZKJBhC0ARIsAJ96n3XK-0y5uKGhO4w7V-A3nvj7WZlIg9NVQ8aeCLYKiEqhcb44rtw9qDoaAmeLEALw_wcB)

## Simple recommender implementation

In [1]:
# Import Pandas
import pandas as pd

# Load Movies Metadata
metadata = pd.read_csv('movies_metadata.csv', low_memory=False)

# Print the first three rows
metadata.head(3)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [ ]:
def top_n_films(data, percentile=0.9, top = 20):
    # Calculate mean of vote average column
    C = data['vote_average'].mean()
    # Calculate the minimum number of votes required to be in the chart, m
    m = data['vote_count'].quantile(percentile)
    # Filter out all qualified movies into a new DataFrame
    data = data[data['vote_count'] >= m].copy()
    # Extract variables used in the formula
    v = data['vote_count']
    R = data['vote_average']
    # Calculation based on the IMDB formula
    data['score'] = (v/(v+m) * R) + (m/(m+v) * C)
    # Sort movies based on score calculated above
    data = data.sort_values('score', ascending=False)

    return data[['title', 'vote_count', 'vote_average', 'score']].head(top)

    

In [25]:
top_n_films(metadata)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


In [86]:
top_n_films(metadata, percentile=0.75, top=10)

,title,vote_count,vote_average,score
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.929668
314,The Shawshank Redemption,8358.0,8.5,8.488324
834,The Godfather,6024.0,8.5,8.483826
40251,Your Name.,1030.0,8.5,8.407913
12481,The Dark Knight,12269.0,8.3,8.292589
2843,Fight Club,9678.0,8.3,8.290612
292,Pulp Fiction,8670.0,8.3,8.289524
39085,Planet Earth,176.0,8.8,8.284853
522,Schindler's List,4436.0,8.3,8.279602
23673,Whiplash,4376.0,8.3,8.279324


## Content based  recommender implementation 

### Plot Description Based Recommender

In [9]:
#Print plot overviews of the first 5 movies.
metadata['overview'].head()


0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: object

In [33]:
#Import TfIdfVectorizer from scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Define a TF-IDF Vectorizer Object. Remove all english stop words such as 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

# Use only the first 5000 rows due to memory limitations
metadata_short = metadata.head(5000).copy()

#Replace NaN with an empty string
metadata_short['overview'] = metadata_short.loc[:, 'overview'].fillna('')

#Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(metadata_short['overview'])

#Output the shape of tfidf_matrix
tfidf_matrix.shape


(5000, 22304)

In [37]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Compute the cosine similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


In [38]:
cosine_sim.shape


(5000, 5000)

In [43]:
#Construct a reverse map of indices and movie titles
indices = pd.Series(metadata_short.index, index=metadata_short['title']).drop_duplicates()


In [70]:
indices.sample(5)


title
Mr. Mom                                   3469
The Big Lebowski                          1650
Romy and Michele's High School Reunion    1447
Haunted Castle                            4328
Playing God                               1569
dtype: int64

In [84]:
def get_recommendations(title, top_n=10):
    # Get the index of the movie that matches the title
    idx = indices[title]
    # Get the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Get the scores of the top_n most similar movies
    sim_scores = sim_scores[1:top_n+1]
    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]
    # Return the top n most similar movies
    return metadata_short['title'].iloc[movie_indices]

In [85]:
get_recommendations('Haunted Castle',15)

4156       Frankie and Johnny
217              Castle Freak
2824                    Gilda
2439      My Boyfriend's Back
2005            The Outsiders
3102               Night Tide
4157       Frankie and Johnny
3317              Cool as Ice
4416          Short Circuit 2
588      Beauty and the Beast
4383             High Spirits
1836        On the Waterfront
1377               Underworld
2488    The Mighty Peking Man
1233       Young Frankenstein
Name: title, dtype: object